In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/tmdb_movies_clean.csv")

df.shape

(8985, 7)

In [2]:
X = df["overview"]
y = df["genres"]

In [3]:
X.shape, y.shape

((8985,), (8985,))

In [4]:
print("Text:", X.iloc[0])
print("Genres:", y.iloc[0])

Text: Fighting crime full-time as Spider-Man in a world that doesn't remember him—and the pressure of seeing his old friends move on without him—sparks a change in Peter Parker he may not have the power to control. But that transformation might also be the only thing that can stop a shocking new threat to the city and those he loves - a powerful villain no one can even see.
Genres: Action|Adventure|Science Fiction


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

In [8]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

In [9]:
X_train.shape, X_val.shape, X_test.shape

((6289,), (1348,), (1348,))

In [10]:
y_train.shape, y_val.shape, y_test.shape

((6289,), (1348,), (1348,))

In [11]:
def get_genre_counts(series):
    return (
        series
        .str.split("|")
        .explode()
        .value_counts()
    )

In [12]:
train_genres = get_genre_counts(y_train)
val_genres = get_genre_counts(y_val)
test_genres = get_genre_counts(y_test)

In [13]:
genre_distribution = pd.DataFrame({
    "Train": train_genres,
    "Validation": val_genres,
    "Test": test_genres
}).fillna(0)

genre_distribution

,Train,Validation,Test
genres,,,
Action,1745,376,389
Adventure,1064,255,222
Animation,715,151,140
Comedy,1800,352,366
Crime,1078,234,234
Documentary,161,36,35
Drama,2643,612,590
Family,646,155,131
Fantasy,726,140,159


In [14]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [15]:
sample = X_train.iloc[0]

print("Before:")
print(sample)

print("\nAfter:")
print(clean_text(sample))

Before:
1934. Private detective Miranda Green investigates a murder perpetrated in the British Embassy in Cairo, where a top secret document was stolen, risking to jeopardize both Buckingham Palace and the peace of the world. All those present in this closed place are suspected: the American photographer, the English student, the American actress, the Egyptian security guard, the ambassador interpreter, the Egyptian gardener and - why not? — the Ambassador himself. But who would have expected that a small group of Nazis would be behind a plot, risking to jeopardize both Buckingham Palace and the peace of the world?

After:
1934 private detective miranda green investigates a murder perpetrated in the british embassy in cairo where a top secret document was stolen risking to jeopardize both buckingham palace and the peace of the world all those present in this closed place are suspected the american photographer the english student the american actress the egyptian security guard the amb

In [16]:
X_train_clean = X_train.apply(clean_text)
X_val_clean = X_val.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

In [17]:
print(X_train_clean.iloc[0])

1934 private detective miranda green investigates a murder perpetrated in the british embassy in cairo where a top secret document was stolen risking to jeopardize both buckingham palace and the peace of the world all those present in this closed place are suspected the american photographer the english student the american actress the egyptian security guard the ambassador interpreter the egyptian gardener and why not the ambassador himself but who would have expected that a small group of nazis would be behind a plot risking to jeopardize both buckingham palace and the peace of the world


In [18]:
def tokenize(text):
    return text.split()

In [19]:
tokens = tokenize(X_train_clean.iloc[0])

print(tokens)
print("Number of tokens:", len(tokens))

['1934', 'private', 'detective', 'miranda', 'green', 'investigates', 'a', 'murder', 'perpetrated', 'in', 'the', 'british', 'embassy', 'in', 'cairo', 'where', 'a', 'top', 'secret', 'document', 'was', 'stolen', 'risking', 'to', 'jeopardize', 'both', 'buckingham', 'palace', 'and', 'the', 'peace', 'of', 'the', 'world', 'all', 'those', 'present', 'in', 'this', 'closed', 'place', 'are', 'suspected', 'the', 'american', 'photographer', 'the', 'english', 'student', 'the', 'american', 'actress', 'the', 'egyptian', 'security', 'guard', 'the', 'ambassador', 'interpreter', 'the', 'egyptian', 'gardener', 'and', 'why', 'not', 'the', 'ambassador', 'himself', 'but', 'who', 'would', 'have', 'expected', 'that', 'a', 'small', 'group', 'of', 'nazis', 'would', 'be', 'behind', 'a', 'plot', 'risking', 'to', 'jeopardize', 'both', 'buckingham', 'palace', 'and', 'the', 'peace', 'of', 'the', 'world']
Number of tokens: 96


In [20]:
train_tokens = X_train_clean.apply(tokenize)

In [21]:
from collections import Counter

word_counts = Counter()

for tokens in train_tokens:
    word_counts.update(tokens)

In [22]:
word_counts.most_common(20)

[('the', 15929),
 ('a', 12787),
 ('to', 9316),
 ('and', 8457),
 ('of', 7923),
 ('in', 5277),
 ('his', 4505),
 ('is', 3662),
 ('s', 3239),
 ('with', 2831),
 ('her', 2537),
 ('he', 2388),
 ('an', 2316),
 ('for', 2099),
 ('on', 2052),
 ('their', 1900),
 ('that', 1826),
 ('as', 1792),
 ('by', 1780),
 ('when', 1696)]

In [23]:
len(word_counts)

23657

In [24]:
sum(1 for count in word_counts.values() if count == 1)

10490

In [25]:
sum(1 for count in word_counts.values() if count < 3)

14355

In [26]:
vocab_words = [word for word, count in word_counts.items() if count >= 2]

len(vocab_words)

13167

In [27]:
word_to_idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for idx, word in enumerate(vocab_words, start=2):
    word_to_idx[word] = idx

In [28]:
len(word_to_idx)

13169

In [31]:
print(list(word_to_idx.items())[:20])

[('<PAD>', 0), ('<UNK>', 1), ('1934', 2), ('private', 3), ('detective', 4), ('miranda', 5), ('green', 6), ('investigates', 7), ('a', 8), ('murder', 9), ('perpetrated', 10), ('in', 11), ('the', 12), ('british', 13), ('embassy', 14), ('cairo', 15), ('where', 16), ('top', 17), ('secret', 18), ('document', 19)]


In [32]:
def encode_tokens(tokens):
    return [word_to_idx.get(token, word_to_idx["<UNK>"]) for token in tokens]

In [33]:
tokens = tokenize(X_train_clean.iloc[0])
encoded = encode_tokens(tokens)

print("Tokens:")
print(tokens)

print("\nEncoded:")
print(encoded)

Tokens:
['1934', 'private', 'detective', 'miranda', 'green', 'investigates', 'a', 'murder', 'perpetrated', 'in', 'the', 'british', 'embassy', 'in', 'cairo', 'where', 'a', 'top', 'secret', 'document', 'was', 'stolen', 'risking', 'to', 'jeopardize', 'both', 'buckingham', 'palace', 'and', 'the', 'peace', 'of', 'the', 'world', 'all', 'those', 'present', 'in', 'this', 'closed', 'place', 'are', 'suspected', 'the', 'american', 'photographer', 'the', 'english', 'student', 'the', 'american', 'actress', 'the', 'egyptian', 'security', 'guard', 'the', 'ambassador', 'interpreter', 'the', 'egyptian', 'gardener', 'and', 'why', 'not', 'the', 'ambassador', 'himself', 'but', 'who', 'would', 'have', 'expected', 'that', 'a', 'small', 'group', 'of', 'nazis', 'would', 'be', 'behind', 'a', 'plot', 'risking', 'to', 'jeopardize', 'both', 'buckingham', 'palace', 'and', 'the', 'peace', 'of', 'the', 'world']

Encoded:
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 11, 15, 16, 8, 17, 18, 19, 20, 21, 22, 23, 24, 25, 

In [34]:
train_token_lengths = train_tokens.apply(len)

In [35]:
train_token_lengths.describe()

count    6289.000000
mean       46.792972
std        25.184695
min         2.000000
25%        28.000000
50%        41.000000
75%        61.000000
max       193.000000
Name: overview, dtype: float64

In [36]:
train_token_lengths.quantile([0.50, 0.75, 0.90, 0.95, 0.99])

0.50     41.00
0.75     61.00
0.90     79.00
0.95     95.00
0.99    128.12
Name: overview, dtype: float64

In [37]:
MAX_LEN = 100

def encode_and_pad(tokens, max_len=MAX_LEN):
    encoded = encode_tokens(tokens)

    if len(encoded) > max_len:
        encoded = encoded[:max_len]

    else:
        encoded = encoded + [word_to_idx["<PAD>"]] * (max_len - len(encoded))

    return encoded

In [38]:
sample_encoded = encode_and_pad(tokenize(X_train_clean.iloc[0]))

print(sample_encoded)
print("Length:", len(sample_encoded))

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 11, 15, 16, 8, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 12, 29, 30, 12, 31, 32, 33, 34, 11, 35, 36, 37, 38, 39, 12, 40, 41, 12, 42, 43, 12, 40, 44, 12, 45, 46, 47, 12, 48, 49, 12, 45, 50, 28, 51, 52, 12, 48, 53, 54, 55, 56, 57, 58, 59, 8, 60, 61, 30, 62, 56, 63, 64, 8, 65, 22, 23, 24, 25, 26, 27, 28, 12, 29, 30, 12, 31, 0, 0, 0, 0]
Length: 100


In [39]:
len(tokenize(X_train_clean.iloc[0]))

96

In [40]:
sum(1 for x in sample_encoded if x == 0)

4

In [41]:
X_train_encoded = X_train_clean.apply(encode_and_pad)

In [42]:
X_val_encoded = X_val_clean.apply(encode_and_pad)

In [43]:
X_test_encoded = X_test_clean.apply(encode_and_pad)

In [44]:
X_train_encoded.shape, X_val_encoded.shape, X_test_encoded.shape

((6289,), (1348,), (1348,))

In [45]:
len(X_train_encoded.iloc[0])

100

In [46]:
print(X_train_encoded.iloc[0])

[2574, 546, 2750, 6472, 1, 6596, 6582, 1213, 6863, 8, 89, 2670, 1, 2472, 2670, 89, 2670, 460, 89, 1213, 6863, 2670, 1, 4882, 1213, 6582, 8, 5252, 2472, 8, 1, 1531, 6582, 2670, 2670, 5252, 1, 1213, 5252, 6863, 2670, 107, 89, 1213, 1531, 8, 89, 2670, 107, 1, 8, 1, 4882, 5846, 6582, 2472, 2670, 6582, 1, 6596, 2670, 6582, 6596, 2670, 89, 6582, 8, 89, 2670, 2472, 1, 1213, 5252, 1, 89, 10699, 2670, 1, 5248, 6582, 1213, 89, 1213, 107, 10699, 1, 2670, 4882, 5248, 8, 107, 107, 7136, 1, 1213, 5252, 1, 460, 8, 1213]


In [47]:
all_genres = sorted(
    set(
        genre
        for genres in y_train
        for genre in genres.split("|")
    )
)

print(all_genres)
print("Number of genres:", len(all_genres))

['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Number of genres: 19


In [48]:
genre_to_idx = {
    genre: idx
    for idx, genre in enumerate(all_genres)
}

genre_to_idx

{'Action': 0,
 'Adventure': 1,
 'Animation': 2,
 'Comedy': 3,
 'Crime': 4,
 'Documentary': 5,
 'Drama': 6,
 'Family': 7,
 'Fantasy': 8,
 'History': 9,
 'Horror': 10,
 'Music': 11,
 'Mystery': 12,
 'Romance': 13,
 'Science Fiction': 14,
 'TV Movie': 15,
 'Thriller': 16,
 'War': 17,
 'Western': 18}

In [49]:
import numpy as np

def encode_genres(genre_string):
    vector = np.zeros(len(all_genres), dtype=np.float32)

    for genre in genre_string.split("|"):
        vector[genre_to_idx[genre]] = 1.0

    return vector

In [50]:
sample_genre = y_train.iloc[0]

print("Original:", sample_genre)
print("Encoded:", encode_genres(sample_genre))

Original: Action|Mystery|Thriller
Encoded: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0.]


In [51]:
len(encode_genres(y_train.iloc[0]))

19

In [52]:
encode_genres(y_train.iloc[0]).sum()

np.float32(3.0)

In [53]:
y_train_encoded = np.array(
    y_train.apply(encode_genres).tolist()
)

In [54]:
y_val_encoded = np.array(
    y_val.apply(encode_genres).tolist()
)

In [55]:
y_test_encoded = np.array(
    y_test.apply(encode_genres).tolist()
)

In [56]:
y_train_encoded.shape, y_val_encoded.shape, y_test_encoded.shape

((6289, 19), (1348, 19), (1348, 19))

In [57]:
X_train_array = np.array(X_train_encoded.tolist(), dtype=np.int64)
X_val_array = np.array(X_val_encoded.tolist(), dtype=np.int64)
X_test_array = np.array(X_test_encoded.tolist(), dtype=np.int64)

In [58]:
X_train_array.shape, X_val_array.shape, X_test_array.shape

((6289, 100), (1348, 100), (1348, 100))

In [59]:
import torch

X_train_tensor = torch.tensor(X_train_array, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_array, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_array, dtype=torch.long)

y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_encoded, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.float32)

In [60]:
print(X_train_tensor.shape)
print(y_train_tensor.shape)

print(X_train_tensor.dtype)
print(y_train_tensor.dtype)

torch.Size([6289, 100])
torch.Size([6289, 19])
torch.int64
torch.float32


In [62]:
from torch.utils.data import Dataset, DataLoader

In [63]:
class MovieGenreDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [64]:
train_dataset = MovieGenreDataset(X_train_tensor, y_train_tensor)
val_dataset = MovieGenreDataset(X_val_tensor, y_val_tensor)
test_dataset = MovieGenreDataset(X_test_tensor, y_test_tensor)

In [65]:
sample_X, sample_y = train_dataset[0]

print("Input:", sample_X)
print("Input shape:", sample_X.shape)

print("\nTarget:", sample_y)
print("Target shape:", sample_y.shape)

Input: tensor([ 2574,   546,  2750,  6472,     1,  6596,  6582,  1213,  6863,     8,
           89,  2670,     1,  2472,  2670,    89,  2670,   460,    89,  1213,
         6863,  2670,     1,  4882,  1213,  6582,     8,  5252,  2472,     8,
            1,  1531,  6582,  2670,  2670,  5252,     1,  1213,  5252,  6863,
         2670,   107,    89,  1213,  1531,     8,    89,  2670,   107,     1,
            8,     1,  4882,  5846,  6582,  2472,  2670,  6582,     1,  6596,
         2670,  6582,  6596,  2670,    89,  6582,     8,    89,  2670,  2472,
            1,  1213,  5252,     1,    89, 10699,  2670,     1,  5248,  6582,
         1213,    89,  1213,   107, 10699,     1,  2670,  4882,  5248,     8,
          107,   107,  7136,     1,  1213,  5252,     1,   460,     8,  1213])
Input shape: torch.Size([100])

Target: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0.,
        0.])
Target shape: torch.Size([19])
